In [3]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist

In [4]:
df = pd.read_csv('video_index.csv')

meta_cols  = ['video_id', 'title', 'datetime', 'transcript']
emb_cols   = [col for col in df.columns if col.startswith('emb_')]

meta       = df[meta_cols].copy()
embeddings = df[emb_cols].values

MODEL_NAME = 'all-mpnet-base-v2'
model      = SentenceTransformer(MODEL_NAME)

print(f"Videos loaded   : {len(meta)}")
print(f"Embedding shape : {embeddings.shape}")
print(f"Model loaded    : {MODEL_NAME}")
meta.head()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Videos loaded   : 123
Embedding shape : (123, 768)
Model loaded    : all-mpnet-base-v2


,video_id,title,datetime,transcript
0,2KaAHD2TOos,Is dust really people?,2026-02-13,(finger squeaking) - Is dust just people? You ...
1,iT2n41ZFDc0,The Intelligence Test Where Ants Beat Humans,2026-02-05,- Thank you to AnyDesk for supporting PBS. Eve...
2,UI8pp2j2fZ8,What time is it on Voyager 1? 🤔,2026-01-30,- What's the farthest manmade object from Eart...
3,rTvTsvT7y7s,What& 39;s REALLY killing all the birds?,2026-01-23,"- You may have gotten wind, the turbines like ..."
4,AxE13l_rXOI,How much does Will Smith know about nature?,2026-01-18,We recently had the opportunity to play a fun ...


In [5]:
query = "What is Dust?"

query_embedding = model.encode([query])

print(f"Query           : {query}")
print(f"Embedding shape : {query_embedding.shape}")

Query           : What is Dust?
Embedding shape : (1, 768)


In [6]:
cosine_scores    = cosine_similarity(query_embedding, embeddings)[0]
euclidean_scores = -cdist(query_embedding, embeddings, metric='euclidean')[0]
manhattan_scores = -cdist(query_embedding, embeddings, metric='cityblock')[0]

print(f"Cosine scores range   : {cosine_scores.min():.3f} to {cosine_scores.max():.3f}")
print(f"Euclidean scores range: {euclidean_scores.min():.3f} to {euclidean_scores.max():.3f}")
print(f"Manhattan scores range: {manhattan_scores.min():.3f} to {manhattan_scores.max():.3f}")


Cosine scores range   : -0.105 to 0.809
Euclidean scores range: -1.463 to -0.602
Manhattan scores range: -31.494 to -12.773


In [7]:
final_scores = cosine_scores

print(f"Final score range: {final_scores.min():.3f} to {final_scores.max():.3f}")

Final score range: -0.105 to 0.809


In [8]:
TOP_K = 5

ranked_indices = np.argsort(final_scores)[::-1][:TOP_K]

results = meta.iloc[ranked_indices][['video_id', 'title']].copy()
results['score'] = final_scores[ranked_indices].round(3)
results['rank']  = range(1, TOP_K + 1)
results = results[['rank', 'video_id', 'title', 'score']].reset_index(drop=True)

print(f"Query : {query}")
print(f"Top-{TOP_K} Results:\n")
results

Query : What is Dust?
Top-5 Results:



,rank,video_id,title,score
0,1,2KaAHD2TOos,Is dust really people?,0.809
1,2,iBkCJBK1N5g,The Secret Ingredient Hiding in Masters Golf S...,0.329
2,3,tp9BQ88rNso,Why NASA Punched an Asteroid,0.301
3,4,uJcXCdbm77g,Space is Full of Junk. Here’s How to Clean It Up…,0.287
4,5,jDr7II5XJ5Q,NASA found something STRANGE in these asteroid...,0.250


In [9]:
THRESHOLD = 0.5

ranked_all    = np.argsort(final_scores)[::-1]
filtered_mask = final_scores[ranked_all] >= THRESHOLD
filtered_idx  = ranked_all[filtered_mask][:TOP_K]

if len(filtered_idx) == 0:
    print(f"No results found above threshold {THRESHOLD}")
else:
    filtered_results = meta.iloc[filtered_idx][['video_id', 'title']].copy()
    filtered_results['score'] = final_scores[filtered_idx].round(3)
    filtered_results['rank']  = range(1, len(filtered_idx) + 1)
    filtered_results = filtered_results[['rank', 'video_id', 'title', 'score']].reset_index(drop=True)

    print(f"Query                  : {query}")
    print(f"Threshold              : {THRESHOLD}")
    print(f"Results above threshold: {len(filtered_idx)}\n")
    filtered_results

Query                  : What is Dust?
Threshold              : 0.5
Results above threshold: 1



In [10]:
def returnSearchResults(query, top_k=5, threshold=0.3):
    # Step 1: Encode query
    query_embedding = model.encode([query])

    # Step 2: Compute cosine similarity
    scores = cosine_similarity(query_embedding, embeddings)[0]

    # Step 3: Rank all videos
    ranked_all = np.argsort(scores)[::-1]

    # Step 4: Apply threshold
    mask        = scores[ranked_all] >= threshold
    filtered    = ranked_all[mask][:top_k]

    # Step 5: Return results
    if len(filtered) == 0:
        print(f"No results found for: '{query}' above threshold {threshold}")
        return pd.DataFrame()

    results = meta.iloc[filtered][['video_id', 'title']].copy()
    results['score']     = scores[filtered].round(3)
    results['rank']      = range(1, len(filtered) + 1)
    results = results[['rank', 'video_id', 'title', 'score']].reset_index(drop=True)
    return results

print("Search function defined successfully!")

Search function defined successfully!


In [11]:
sample_queries = [
    "How do human brain works?",
    "What happens to an astronaut's body in space?",
    "Do whales communicate?"
]

for q in sample_queries:
    print(f"\n{'='*60}")
    print(f"Query: {q}")
    print(f"{'='*60}")
    result = returnSearchResults(q)
    if not result.empty:
        print(result.to_string(index=False))


Query: How do human brain works?
 rank    video_id                                                     title  score
    1 YSM13kIpV4Q The most detailed brain map EVER will blow your mind! ￼🧠🤯  0.623
    2 pPIem63bC4w             The Weird Reason Some People Can Taste Colors  0.429
    3 iT2n41ZFDc0              The Intelligence Test Where Ants Beat Humans  0.388
    4 L4zFS0RPP7c                               Why You See Faces in Things  0.351
    5 vRqCs2SUdxY                       The Real (Weird) Way We See Numbers  0.338

Query: What happens to an astronaut's body in space?
 rank    video_id                                          title  score
    1 plq-2zw2aIE This is why astronauts look so WEIRD in zero g  0.635
    2 hyarxb96lgM            Where does lost weight ACTUALLY go?  0.366

Query: Do whales communicate?
 rank    video_id                                                          title  score
    1 4mTxc_v__Ts                        Did we just decode the whale alphabet?

In [12]:
eval_queries = pd.read_csv('search_queries.csv')

correct_top1 = 0
correct_top3 = 0
correct_top5 = 0
rank_list    = []

for _, row in eval_queries.iterrows():
    q           = row['query']
    expected_id = row['relevant_video_id']

    query_emb = model.encode([q])
    scores    = cosine_similarity(query_emb, embeddings)[0]
    ranked    = np.argsort(scores)[::-1]
    ranked_ids = meta['video_id'].iloc[ranked].tolist()

    rank = ranked_ids.index(expected_id) + 1 if expected_id in ranked_ids else len(ranked_ids) + 1

    if rank == 1: correct_top1 += 1
    if rank <= 3: correct_top3 += 1
    if rank <= 5: correct_top5 += 1
    rank_list.append(rank)

n = len(eval_queries)
print(f"Total queries  : {n}")
print(f"Top-1 Recall   : {round(correct_top1/n, 3)}")
print(f"Top-3 Recall   : {round(correct_top3/n, 3)}")
print(f"Top-5 Recall   : {round(correct_top5/n, 3)}")
print(f"Avg Rank       : {round(np.mean(rank_list), 2)}")

Total queries  : 83
Top-1 Recall   : 0.952
Top-3 Recall   : 0.988
Top-5 Recall   : 0.988
Avg Rank       : 2.52
